<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/sequence-models/seq2seq_encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kaybolan Gradyan, Kapılı Hücreler ve Seq2Seq (Encoder–Decoder)

Bu defter `rnn_pytorch_keras`, `lstm_pytorch_keras` ve `gru_pytorch_keras` defterlerinin devamıdır.

1. **Kaybolan gradyan problemini** deneysel olarak ölçüyoruz: RNN vs LSTM vs GRU
2. LSTM ve GRU kapılarının bu sorunu nasıl çözdüğünü formüllerle özetliyoruz
3. **Seq2Seq** (Encoder–Decoder) mimarisini sıfırdan kurup bir tarih dönüştürme görevi öğretiyoruz: `"12 ocak 1998"` → `"12.01.1998"`
4. **Attention** ekleyip darboğaz (bottleneck) problemini çözüyoruz ve hizalamayı görselleştiriyoruz

> Colab'da `Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU` seçerseniz eğitim daha hızlı olur, ama CPU'da da birkaç dakikada çalışır.

In [ ]:
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(42); np.random.seed(42); torch.manual_seed(42)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

---
## 1) Kaybolan gradyan (vanishing gradient)

Basit RNN'de gizli durum $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$ şeklinde güncellenir. Zincir kuralıyla $T$ adım geriye giden gradyan:

$$\frac{\partial h_T}{\partial h_1} = \prod_{t=2}^{T} \underbrace{\text{diag}\left(1 - h_t^2\right) W_h}_{\text{her adımda çarpılan matris}}$$

Bu çarpımın normu her adımda 1'den küçükse gradyan **üstel olarak sıfıra gider** (kaybolan gradyan); 1'den büyükse **patlar** (exploding gradient). Sonuç: RNN uzak geçmişteki bilgiyi öğrenemez.

In [ ]:
for carpan in [0.9, 0.5, 1.1]:
    print(f"{carpan}^50 = {carpan ** 50:.2e}    {carpan}^100 = {carpan ** 100:.2e}")

### Deney: son çıktının, ilk zaman adımlarındaki girdilere göre gradyanı

Uzunluğu 100 olan rastgele bir dizi veriyoruz ve **sadece son adımın** çıktısından bir kayıp hesaplıyoruz. Sonra her zaman adımındaki girdiye göre gradyanın büyüklüğünü ölçüyoruz: $\left\lVert \partial \mathcal{L} / \partial x_t \right\rVert$.

In [ ]:
def gradyan_profili(hucre_tipi, T=100, girdi=16, gizli=64, tekrar=10):
    profiller = []
    for s in range(tekrar):
        torch.manual_seed(s)
        model = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[hucre_tipi](girdi, gizli, batch_first=True)
        if hucre_tipi == "LSTM":   # yaygın pratik: unutma kapısı bias'ını 1 ile başlat
            with torch.no_grad():
                for isim, p in model.named_parameters():
                    if "bias" in isim:
                        p[gizli:2 * gizli].fill_(1.0)
        x = torch.randn(1, T, girdi, requires_grad=True)
        cikti, _ = model(x)
        cikti[:, -1].sum().backward()
        profiller.append(x.grad[0].norm(dim=-1).numpy())
    return np.mean(profiller, axis=0)

plt.figure(figsize=(10, 4.5))
for tip in ["RNN", "LSTM", "GRU"]:
    plt.semilogy(gradyan_profili(tip), label=tip, lw=2)
plt.xlabel("zaman adımı t  (99 = son adım)")
plt.ylabel(r"$\|\partial \mathcal{L} / \partial x_t\|$  (log ölçek)")
plt.title("Son adımdan geriye akan gradyanın büyüklüğü")
plt.legend(); plt.grid(alpha=0.3); plt.show()

RNN'de gradyan birkaç düzine adım geriye gidince pratikte **sıfırlanıyor**. LSTM ve GRU'da çok daha yavaş azalıyor; bu da onların uzun bağımlılıkları öğrenebilmesinin sebebi.

---
## 2) Kapılar sorunu nasıl çözüyor?

### LSTM: 3 kapı + taşıma bandı (cell state)

$$
\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) && \text{Unutma kapısı: eski hafızanın ne kadarı kalsın?}\\
i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) && \text{Giriş kapısı: yeni bilginin ne kadarı yazılsın?}\\
\tilde c_t &= \tanh(W_c [h_{t-1}, x_t] + b_c) && \text{Aday hafıza}\\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t && \textbf{Taşıma bandı: toplamsal güncelleme}\\
o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) && \text{Çıkış kapısı: hafızanın ne kadarı dışarı verilsin?}\\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

Kritik nokta: $\frac{\partial c_t}{\partial c_{t-1}} = f_t$. Hafıza **çarpımsal bir matris** yerine **eleman bazlı bir kapı** ile taşınıyor. $f_t \approx 1$ olduğunda gradyan bant boyunca neredeyse kayıpsız akar.

### GRU: 2 kapı, tek durum

$$
\begin{aligned}
z_t &= \sigma(W_z [h_{t-1}, x_t]) && \text{Güncelleme kapısı (LSTM'in unutma + giriş kapısının birleşimi)}\\
r_t &= \sigma(W_r [h_{t-1}, x_t]) && \text{Sıfırlama kapısı: aday hesaplarken geçmişin ne kadarına bakılsın?}\\
\tilde h_t &= \tanh(W [r_t \odot h_{t-1}, x_t])\\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t
\end{aligned}
$$

| | RNN | LSTM | GRU |
|---|---|---|---|
| Kapı sayısı | 0 | 3 (giriş, unutma, çıkış) | 2 (güncelleme, sıfırlama) |
| Durum | $h$ | $h$ + $c$ (taşıma bandı) | $h$ |
| Parametre (aynı gizli boyutta) | 1× | 4× | 3× |
| Uzun bağımlılık | Zayıf | Çok iyi | Çok iyi |

In [ ]:
for tip in [nn.RNN, nn.LSTM, nn.GRU]:
    m = tip(input_size=64, hidden_size=128)
    print(f"{tip.__name__:5s}: {sum(p.numel() for p in m.parameters()):,} parametre")

---
## 3) Seq2Seq: Encoder–Decoder mimarisi (Sutskever vd., 2014)

Girdi ve çıktı uzunluklarının **farklı** olabildiği problemler (çeviri, özetleme, soru–cevap) için:

```
           ENCODER                               DECODER
 "1 2   o c a k   1 9 9 8"              <sos>  1   2   .   0   1  ...
   │ │ │ │ │ │ │ │ │ │ │ │                 │    │   │   │   │   │
  [GRU→GRU→GRU→ ... →GRU] ──── h (bağlam vektörü) ────▶ [GRU→GRU→GRU→GRU→ ...]
                                                     │    │   │   │   │
                                                     1    2   .   0   1  ... <eos>
```

- **Encoder** girdiyi okur ve tüm bilgiyi **tek bir sabit boyutlu vektöre** (son gizli durum) sıkıştırır.
- **Decoder** bu vektörle başlar ve token token çıktı üretir; her adımda bir önceki çıktıyı girdi olarak alır.
- **Teacher forcing:** Eğitimde decoder'a kendi tahmini yerine **doğru** önceki token verilir → hızlı ve kararlı eğitim.
- **Çıkarım:** Doğru cevap bilinmediği için decoder kendi ürettiği tokenı bir sonraki adıma besler (greedy decoding).

### Veri: Türkçe tarih → sayısal format

In [ ]:
AYLAR = ["ocak", "şubat", "mart", "nisan", "mayıs", "haziran",
         "temmuz", "ağustos", "eylül", "ekim", "kasım", "aralık"]
GUNLER = ["pazartesi", "salı", "çarşamba", "perşembe", "cuma", "cumartesi", "pazar"]

def rastgele_tarih():
    gun, ay, yil = random.randint(1, 28), random.randint(1, 12), random.randint(1950, 2049)
    bicim = random.random()
    if bicim < 0.4:
        girdi = f"{gun} {AYLAR[ay - 1]} {yil}"
    elif bicim < 0.7:
        girdi = f"{gun} {AYLAR[ay - 1]} {yil} {random.choice(GUNLER)}"
    elif bicim < 0.85:
        girdi = f"{gun:02d} {AYLAR[ay - 1].upper()} {yil}"
    else:
        girdi = f"{AYLAR[ay - 1]} {gun}, {yil}"
    return girdi, f"{gun:02d}.{ay:02d}.{yil}"

for _ in range(6):
    print("%-35s → %s" % rastgele_tarih())

In [ ]:
PAD, SOS, EOS = 0, 1, 2
karakterler = sorted(set("".join(a + b for a, b in (rastgele_tarih() for _ in range(50_000)))))
itos = ["<pad>", "<sos>", "<eos>"] + karakterler
stoi = {c: i for i, c in enumerate(itos)}
SOZLUK = len(itos)
CIKTI_UZ = 10 + 1   # "dd.mm.yyyy" + <eos>

def kodla(metin, eos=False):
    return [stoi[c] for c in metin] + ([EOS] if eos else [])

def yigin_uret(n):
    ciftler = [rastgele_tarih() for _ in range(n)]
    girdiler = [kodla(g) for g, _ in ciftler]
    maks = max(map(len, girdiler))
    X = torch.full((n, maks), PAD, dtype=torch.long)
    for i, g in enumerate(girdiler):
        X[i, :len(g)] = torch.tensor(g)
    Y = torch.tensor([kodla(c, eos=True) for _, c in ciftler])
    return X.to(cihaz), Y.to(cihaz), ciftler

print("Sözlük boyutu:", SOZLUK)
X_ornek, Y_ornek, _ = yigin_uret(3)
print("Girdi tensörü:", X_ornek.shape, "| Hedef tensörü:", Y_ornek.shape)

### Model: Encoder + Decoder (attention'sız)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, sozluk, d_emb=64, gizli=128):
        super().__init__()
        self.emb = nn.Embedding(sozluk, d_emb, padding_idx=PAD)
        self.gru = nn.GRU(d_emb, gizli, batch_first=True, bidirectional=True)
        self.koprule = nn.Linear(2 * gizli, gizli)   # çift yönlü durumu decoder boyutuna indir

    def forward(self, x):
        uzunluk = (x != PAD).sum(1).cpu()
        paket = nn.utils.rnn.pack_padded_sequence(self.emb(x), uzunluk, batch_first=True, enforce_sorted=False)
        ciktilar, h = self.gru(paket)
        ciktilar, _ = nn.utils.rnn.pad_packed_sequence(ciktilar, batch_first=True, total_length=x.size(1))
        h = torch.tanh(self.koprule(torch.cat([h[0], h[1]], dim=-1))).unsqueeze(0)   # (1, B, gizli)
        return ciktilar, h                                                            # (B, T, 2*gizli)


class Decoder(nn.Module):
    def __init__(self, sozluk, d_emb=64, gizli=128):
        super().__init__()
        self.emb = nn.Embedding(sozluk, d_emb)
        self.gru = nn.GRU(d_emb, gizli, batch_first=True)
        self.cikis = nn.Linear(gizli, sozluk)

    def adim(self, token, h, enc_ciktilar=None, enc_maske=None):
        o, h = self.gru(self.emb(token).unsqueeze(1), h)
        return self.cikis(o.squeeze(1)), h, None


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder

    def forward(self, x, y):
        """Teacher forcing ile eğitim: decoder'a her adımda DOĞRU önceki token verilir."""
        enc_ciktilar, h = self.encoder(x)
        maske = x != PAD
        token = torch.full((x.size(0),), SOS, device=x.device)
        logitler = []
        for t in range(y.size(1)):
            logit, h, _ = self.decoder.adim(token, h, enc_ciktilar, maske)
            logitler.append(logit)
            token = y[:, t]
        return torch.stack(logitler, 1)

    @torch.no_grad()
    def uret(self, x, maks_uz=CIKTI_UZ):
        """Greedy decoding: her adımda en olası tokenı seç ve bir sonraki adıma ver."""
        enc_ciktilar, h = self.encoder(x)
        maske = x != PAD
        token = torch.full((x.size(0),), SOS, device=x.device)
        ciktilar, dikkatler = [], []
        for _ in range(maks_uz):
            logit, h, dikkat = self.decoder.adim(token, h, enc_ciktilar, maske)
            token = logit.argmax(-1)
            ciktilar.append(token)
            dikkatler.append(dikkat)
        return torch.stack(ciktilar, 1), dikkatler

### Model: Attention'lı Decoder (Bahdanau / Luong)

Attention'sız Seq2Seq'te tüm girdi **tek bir vektöre** sıkıştırılır; uzun girdilerde bu bir **darboğazdır**. Attention ile decoder her adımda encoder'ın **tüm** çıktılarına bakar ve hangisine odaklanacağını öğrenir:

$$
e_{t,s} = \mathbf{h}_t^\top W_a \bar{\mathbf{h}}_s, \qquad
\alpha_{t,s} = \frac{\exp(e_{t,s})}{\sum_{s'} \exp(e_{t,s'})}, \qquad
\mathbf{c}_t = \sum_s \alpha_{t,s}\, \bar{\mathbf{h}}_s
$$

Bu fikir, birkaç yıl sonra **Transformer**'ın (*"Attention Is All You Need"*, 2017) temel yapı taşı olacaktır.

In [ ]:
class AttentionDecoder(Decoder):
    def __init__(self, sozluk, d_emb=64, gizli=128):
        super().__init__(sozluk, d_emb, gizli)
        self.W_a = nn.Linear(2 * gizli, gizli, bias=False)      # encoder çıktısını decoder uzayına taşı
        self.birlestir = nn.Linear(gizli + 2 * gizli, gizli)
        self.cikis = nn.Linear(gizli, sozluk)

    def adim(self, token, h, enc_ciktilar, enc_maske):
        o, h = self.gru(self.emb(token).unsqueeze(1), h)                       # (B, 1, gizli)
        skor = torch.bmm(self.W_a(enc_ciktilar), o.transpose(1, 2)).squeeze(-1) # (B, T)
        skor = skor.masked_fill(~enc_maske, float("-inf"))                     # pad'lere bakma
        alfa = F.softmax(skor, dim=-1)                                          # (B, T)
        baglam = torch.bmm(alfa.unsqueeze(1), enc_ciktilar).squeeze(1)          # (B, 2*gizli)
        birlesik = torch.tanh(self.birlestir(torch.cat([o.squeeze(1), baglam], -1)))
        return self.cikis(birlesik), h, alfa

### Eğitim

In [ ]:
def egit(model, adim_sayisi=2500, yigin=128, lr=2e-3):
    model.to(cihaz)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    kayiplar = []
    t0 = time.time()
    for adim in range(1, adim_sayisi + 1):
        model.train()
        X, Y, _ = yigin_uret(yigin)
        logit = model(X, Y)
        kayip = F.cross_entropy(logit.reshape(-1, SOZLUK), Y.reshape(-1))
        opt.zero_grad()
        kayip.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)       # patlayan gradyana karşı
        opt.step()
        kayiplar.append(kayip.item())
        if adim % 500 == 0:
            print(f"  adım {adim:5d} | kayıp {np.mean(kayiplar[-100:]):.4f} | {time.time() - t0:.0f} sn")
    return kayiplar

print("Attention'sız Seq2Seq:")
duz_model = Seq2Seq(Encoder(SOZLUK), Decoder(SOZLUK))
duz_kayip = egit(duz_model)

print("\nAttention'lı Seq2Seq:")
dikkat_model = Seq2Seq(Encoder(SOZLUK), AttentionDecoder(SOZLUK))
dikkat_kayip = egit(dikkat_model)

In [ ]:
def yumusat(x, k=50):
    return np.convolve(x, np.ones(k) / k, mode="valid")

plt.figure(figsize=(10, 4))
plt.semilogy(yumusat(duz_kayip), label="Seq2Seq")
plt.semilogy(yumusat(dikkat_kayip), label="Seq2Seq + Attention")
plt.xlabel("adım"); plt.ylabel("cross-entropy (log)")
plt.title("Eğitim kaybı"); plt.legend(); plt.grid(alpha=0.3); plt.show()

### Değerlendirme

In [ ]:
def coz(idler):
    metin = []
    for i in idler:
        if i == EOS:
            break
        metin.append(itos[i])
    return "".join(metin)

def degerlendir(model, n=2000):
    model.eval()
    X, Y, ciftler = yigin_uret(n)
    tahmin, _ = model.uret(X)
    tam_eslesme = np.mean([coz(t) == c for t, (_, c) in zip(tahmin.tolist(), ciftler)])
    karakter = (tahmin[:, :10] == Y[:, :10]).float().mean().item()
    return tam_eslesme, karakter

for isim, m in [("Seq2Seq", duz_model), ("Seq2Seq + Attention", dikkat_model)]:
    tam, kar = degerlendir(m)
    print(f"{isim:22s} → tam eşleşme: %{tam * 100:.1f} | karakter doğruluğu: %{kar * 100:.1f}")

In [ ]:
ornekler = ["3 mart 2024", "29 ekim 1923", "1 ocak 2000 cumartesi", "17 AĞUSTOS 1999",
            "eylül 9, 1971", "14 şubat 2026 cuma", "30 kasım 1985"]

def tek_tahmin(model, metin):
    x = torch.tensor([kodla(metin)], device=cihaz)
    tahmin, dikkat = model.uret(x)
    return coz(tahmin[0].tolist()), dikkat

pd.DataFrame([{"Girdi": s,
               "Seq2Seq": tek_tahmin(duz_model, s)[0],
               "Seq2Seq + Attention": tek_tahmin(dikkat_model, s)[0]} for s in ornekler])

### Attention haritası: model girdinin neresine bakıyor?

Her satır bir çıktı karakteri, her sütun bir girdi karakteri. Gün rakamları üretilirken gün kısmına, ay rakamları üretilirken **ay ismine**, yıl üretilirken yıl rakamlarına odaklanma bekleriz. Bu hizalamayı modele hiç söylemedik; kendisi öğrendi.

In [ ]:
fig, eksenler = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, metin in zip(eksenler, ["29 ekim 1923", "14 şubat 2026 cuma", "eylül 9, 1971"]):
    cikti, dikkat = tek_tahmin(dikkat_model, metin)
    harita = torch.stack(dikkat[:len(cikti)], 0)[:, 0].cpu().numpy()   # (çıktı, girdi)
    ax.imshow(harita, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(metin))); ax.set_xticklabels(list(metin))
    ax.set_yticks(range(len(cikti))); ax.set_yticklabels(list(cikti))
    ax.set_xlabel("girdi"); ax.set_ylabel("çıktı")
    ax.set_title(f"'{metin}' → '{cikti}'")
plt.tight_layout(); plt.show()

---
## Özet

- **Kaybolan gradyan:** RNN'de gradyan, zaman boyunca tekrarlanan matris çarpımları yüzünden üstel olarak küçülür.
- **LSTM** toplamsal hücre durumu (taşıma bandı) ve 3 kapı ile; **GRU** daha az parametreli 2 kapı ile bu sorunu hafifletir.
- **Seq2Seq**, değişken uzunluklu bir diziyi başka bir diziye çevirir: Encoder → bağlam vektörü → Decoder.
- **Teacher forcing** eğitimi hızlandırır; çıkarımda greedy (veya beam search) decoding kullanılır.
- **Attention**, sabit boyutlu bağlam vektörü darboğazını kaldırır ve yorumlanabilir hizalamalar üretir. Bu fikir Transformer mimarisinin kalbidir → `transformers-llm/01_transformer_sifirdan.ipynb`